# Unify Benchmark Toolkit

**Purpose:** Fast, expressive evaluation of time-series forecasting methods on industrial data.

## Models Included
| Model | Type | Covariates | Notes |
|-------|------|------------|-------|
| **Naive** | Baseline | ✗ | Last-value forecast |
| **Seasonal Naive** | Baseline | ✗ | Repeat last season |
| **TabPFN-TS** | Foundation | ✓ | PriorLabs, zero-shot |
| **Chronos-2** | Foundation | ✓ | Amazon, zero-shot |

## Design Principles
- **Small scale:** Fast iterations (<5 min per experiment)
- **Expressive:** Results should transfer to larger scale
- **Modular:** Easy to swap models, datasets, metrics

## Structure
1. **Visual Test:** One sample, plot true vs predicted
2. **Holistic Metrics:** GIFT-eval style aggregated scores
3. **Visualizations:** Bar charts, heatmaps, radar plots, rankings

## Resources
- [GIFT-Eval](https://github.com/SalesforceAIResearch/gift-eval) - Benchmark methodology
- [TabPFN-TS](https://github.com/PriorLabs/tabpfn-time-series) - PriorLabs
- [Chronos-2](https://github.com/amazon-science/chronos-forecasting) - Amazon

---
## Setup

In [1]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, List, Callable, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

print("Core imports ready.")

Core imports ready.


In [ ]:
# Model imports (run separately to isolate errors)
MODELS_AVAILABLE = {}

# TabPFN-TS (time series foundation model)
try:
    from tabpfn_time_series import TabPFNTSPipeline, TabPFNMode
    MODELS_AVAILABLE['tabpfn_ts'] = True
    print("✓ TabPFN-TS available (time series foundation model)")
except ImportError as e:
    MODELS_AVAILABLE['tabpfn_ts'] = False
    print(f"✗ TabPFN-TS not available: {e}")

# Chronos
try:
    from chronos import BaseChronosPipeline
    import torch
    MODELS_AVAILABLE['chronos'] = True
    print("✓ Chronos available")
except ImportError as e:
    MODELS_AVAILABLE['chronos'] = False
    print(f"✗ Chronos not available: {e}")

print(f"\nAvailable models: {[k for k,v in MODELS_AVAILABLE.items() if v]}")

---
## Configuration

In [3]:
@dataclass
class BenchmarkConfig:
    """Benchmark configuration - modify this for experiments."""
    
    # Data
    context_length: int = 256      # History length
    horizons: List[int] = None     # Forecast horizons to test
    
    # Evaluation
    n_samples: int = 10            # Number of test windows (for speed)
    quantiles: List[float] = None  # For probabilistic metrics
    
    # Models to test
    models: List[str] = None
    
    def __post_init__(self):
        self.horizons = self.horizons or [24, 48, 96]
        self.quantiles = self.quantiles or [0.1, 0.5, 0.9]
        self.models = self.models or ['naive', 'tabpfn', 'chronos']

# Default config - fast iterations
CFG = BenchmarkConfig(
    context_length=256,
    horizons=[24, 48],  # Short for speed
    n_samples=5,        # Few samples for speed
)
print(f"Config: {CFG}")

Config: BenchmarkConfig(context_length=256, horizons=[24, 48], n_samples=5, quantiles=[0.1, 0.5, 0.9], models=['naive', 'tabpfn', 'chronos'])


---
## Metrics (GIFT-Eval Style)

Following [GIFT-Eval](https://arxiv.org/abs/2410.10393):

| Metric | What It Measures | Interpretation |
|--------|------------------|----------------|
| **MAE** | Mean Absolute Error | Lower is better (scale-dependent) |
| **rMAE** | Relative MAE vs Naive | **<1 = beats naive**, >1 = worse than naive |
| **MASE** | MAE scaled by series difficulty | <1 = better than seasonal naive *in-sample* |
| **CRPS** | Probabilistic calibration | Lower is better (= MAE for point forecasts) |

**Key distinction:**
- `rMAE` = direct test-time comparison: "Does this model beat naive on *this forecast*?"
- `MASE` = scale-free difficulty: "Is this model better than how hard the series typically is?"

In [ ]:
def compute_mase(actual: np.ndarray, predicted: np.ndarray,
                 history: np.ndarray, seasonality: int = 1) -> float:
    """
    Mean Absolute Scaled Error (Hyndman & Koehler, 2006).
    
    Scaled by IN-SAMPLE seasonal naive error on history.
    Measures: "How hard is this series to forecast?"
    
    NOTE: This is NOT a direct comparison to naive forecast at test time.
    Use rMAE for that.
    """
    mae = np.mean(np.abs(actual - predicted))
    naive_errors = np.abs(history[seasonality:] - history[:-seasonality])
    scale = np.mean(naive_errors)
    return mae / scale if scale > 0 else np.inf


def compute_crps(actual: np.ndarray, quantile_preds: Optional[Dict[float, np.ndarray]]) -> float:
    """
    Continuous Ranked Probability Score (approximate via pinball loss).
    For point forecasts (no quantiles), returns None (handled in compute_all_metrics).
    """
    if quantile_preds is None or len(quantile_preds) == 0:
        return None
    
    crps = 0.0
    quantiles = sorted(quantile_preds.keys())
    for q in quantiles:
        pred_q = quantile_preds[q]
        error = actual - pred_q
        loss = np.where(error >= 0, q * error, (q - 1) * error)
        crps += np.mean(loss)
    return crps / len(quantiles)


def compute_all_metrics(actual: np.ndarray, predicted: np.ndarray,
                        history: np.ndarray, quantile_preds: Optional[Dict] = None,
                        seasonality: int = 24) -> Dict[str, float]:
    """Compute all metrics for a single forecast."""
    mae = mean_absolute_error(actual, predicted)
    
    # Compute naive MAE for rMAE
    naive_pred = np.full_like(actual, history[-1])
    naive_mae = mean_absolute_error(actual, naive_pred)
    
    metrics = {
        'MAE': mae,
        'MSE': mean_squared_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'MASE': compute_mase(actual, predicted, history, seasonality),
        # rMAE: Direct comparison to naive. <1 = beats naive, >1 = worse
        'rMAE': mae / naive_mae if naive_mae > 0 else np.inf,
    }
    
    # Normalized MSE (comparable across datasets)
    metrics['NMSE'] = metrics['MSE'] / np.var(actual) if np.var(actual) > 0 else np.inf
    
    # CRPS: use quantiles if available, else fall back to MAE
    crps = compute_crps(actual, quantile_preds)
    metrics['CRPS'] = crps if crps is not None else mae
    
    return metrics

print("Metrics defined: MAE, MSE, RMSE, MASE, rMAE, NMSE, CRPS")
print("  - rMAE: Direct comparison to naive (rMAE < 1 = better than naive)")
print("  - CRPS: Falls back to MAE for point forecasts")

---
## Model Wrappers

Unified interface for all models: `predict(history, horizon) -> (point, quantiles)`

In [ ]:
class BaseForecaster:
    """Base class for all forecasters."""
    name: str = "base"
    
    def predict(self, history: np.ndarray, horizon: int,
                covariates: Optional[np.ndarray] = None,
                future_covariates: Optional[np.ndarray] = None
               ) -> tuple[np.ndarray, Optional[Dict[float, np.ndarray]]]:
        """Returns (point_forecast, quantile_forecasts)."""
        raise NotImplementedError


class NaiveForecaster(BaseForecaster):
    """Last-value naive forecast. Baseline."""
    name = "naive"
    
    def predict(self, history, horizon, **kwargs):
        point = np.full(horizon, history[-1])
        return point, None


class SeasonalNaiveForecaster(BaseForecaster):
    """Seasonal naive: repeat last season."""
    name = "seasonal_naive"
    
    def __init__(self, seasonality: int = 24):
        self.seasonality = seasonality
    
    def predict(self, history, horizon, **kwargs):
        last_season = history[-self.seasonality:]
        repeats = (horizon // self.seasonality) + 1
        point = np.tile(last_season, repeats)[:horizon]
        return point, None


class TabPFNTSForecaster(BaseForecaster):
    """
    TabPFN-TS: Time series foundation model from PriorLabs.
    
    - Zero-shot forecasting (no training needed)
    - Native covariate support
    - Probabilistic predictions (quantiles)
    """
    name = "tabpfn_ts"
    
    def __init__(self, mode: str = 'CLIENT'):
        if not MODELS_AVAILABLE.get('tabpfn_ts'):
            raise ImportError("TabPFN-TS not installed. Install with: pip install tabpfn-time-series")
        mode_enum = TabPFNMode.CLIENT if mode == 'CLIENT' else TabPFNMode.LOCAL
        self.pipeline = TabPFNTSPipeline(tabpfn_mode=mode_enum)
    
    def predict(self, history, horizon, covariates=None, future_covariates=None):
        n = len(history)
        context_df = pd.DataFrame({
            'item_id': 'series_0',
            'timestamp': pd.date_range('2020-01-01', periods=n, freq='h'),
            'target': history
        })
        
        if covariates is not None:
            for i in range(covariates.shape[1]):
                context_df[f'cov_{i}'] = covariates[:, i]
        
        future_df = None
        if future_covariates is not None:
            future_df = pd.DataFrame({
                'item_id': 'series_0',
                'timestamp': pd.date_range(
                    context_df['timestamp'].iloc[-1] + pd.Timedelta(hours=1),
                    periods=horizon, freq='h'
                )
            })
            for i in range(future_covariates.shape[1]):
                future_df[f'cov_{i}'] = future_covariates[:, i]
        
        if future_df is not None:
            pred_df = self.pipeline.predict_df(context_df=context_df, future_df=future_df)
        else:
            pred_df = self.pipeline.predict_df(context_df=context_df, prediction_length=horizon)
        
        point = pred_df['target'].values
        quantiles = {q: pred_df[q].values for q in [0.1, 0.5, 0.9] if q in pred_df.columns}
        
        return point, quantiles if quantiles else None


class Chronos2Forecaster(BaseForecaster):
    """
    Chronos-2: Time series foundation model from Amazon (2024).
    
    - Zero-shot forecasting
    - Multivariate support (covariates)
    - Probabilistic predictions
    - 8192 context length
    
    Models: amazon/chronos-t5-{tiny,mini,small,base,large}
            amazon/chronos-bolt-{tiny,mini,small,base}
    """
    name = "chronos2"
    
    def __init__(self, model_name: str = "amazon/chronos-bolt-small", device: str = "cpu"):
        if not MODELS_AVAILABLE.get('chronos'):
            raise ImportError("Chronos not installed. Install with: pip install chronos-forecasting")
        
        self.model_name = model_name
        self.device = device
        self.pipeline = BaseChronosPipeline.from_pretrained(
            model_name, device_map=device, torch_dtype=torch.float32
        )
    
    def predict(self, history, horizon, covariates=None, future_covariates=None):
        # Build context tensor - shape: (n_series, context_length)
        # For multivariate: stack target + covariates
        if covariates is not None:
            # Multivariate: target is first channel, covariates follow
            context = np.column_stack([history, covariates])  # (T, 1+n_cov)
            context = torch.tensor(context.T, dtype=torch.float32)  # (1+n_cov, T)
        else:
            context = torch.tensor(history, dtype=torch.float32).unsqueeze(0)  # (1, T)
        
        # Predict
        forecast = self.pipeline.predict(context, horizon, num_samples=100)
        
        # Extract target channel (first one)
        samples = forecast[0].numpy()  # Shape: (num_samples, horizon)
        
        point = np.median(samples, axis=0)
        quantiles = {
            0.1: np.percentile(samples, 10, axis=0),
            0.5: np.percentile(samples, 50, axis=0),
            0.9: np.percentile(samples, 90, axis=0),
        }
        return point, quantiles


# Backward compatibility alias
ChronosForecaster = Chronos2Forecaster


def get_forecaster(name: str) -> BaseForecaster:
    """Factory function to get forecaster by name."""
    forecasters = {
        'naive': NaiveForecaster,
        'seasonal_naive': SeasonalNaiveForecaster,
        'tabpfn_ts': TabPFNTSForecaster,
        'chronos': Chronos2Forecaster,
        'chronos2': Chronos2Forecaster,
    }
    if name not in forecasters:
        raise ValueError(f"Unknown forecaster: {name}. Available: {list(forecasters.keys())}")
    return forecasters[name]()

print("Model wrappers defined:")
print("  - naive, seasonal_naive: Baselines")
print("  - tabpfn_ts: TabPFN-TS (supports covariates)")
print("  - chronos2: Chronos-2 (supports covariates)")

---
## Data Loading

Simple loaders for test datasets. Add new datasets here.

In [ ]:
def load_etth1(data_dir: str = "../data") -> pd.DataFrame:
    """Load ETTh1 dataset."""
    path = Path(data_dir) / "ETTh1.csv"
    if not path.exists():
        # Download if not present
        url = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"
        df = pd.read_csv(url)
        path.parent.mkdir(exist_ok=True)
        df.to_csv(path, index=False)
    else:
        df = pd.read_csv(path)
    
    df['date'] = pd.to_datetime(df['date'])
    return df.rename(columns={'date': 'timestamp', 'OT': 'target'})


def load_synthetic_pendulum(n_links: int = 3, n_samples: int = 5000,
                            noise_std: float = 0.02) -> pd.DataFrame:
    """
    Generate synthetic N-link pendulum data.
    Each link has: angle (theta), angular velocity (omega)
    
    Fixed: omega is computed from clean signal, noise added separately
    to avoid numerical differentiation amplifying noise.
    """
    np.random.seed(42)
    dt = 0.02  # 50 Hz
    t = np.arange(n_samples) * dt
    
    data = {'timestamp': pd.date_range('2020-01-01', periods=n_samples, freq='20ms')}
    
    # Natural frequencies decrease with link number (physics)
    for i in range(n_links):
        omega_n = 2.0 / (i + 1)  # Natural frequency
        phase = np.random.uniform(0, 2*np.pi)
        damping = 0.02 * (i + 1)
        
        # Clean signal (no noise yet)
        envelope = np.exp(-damping * t)
        theta_clean = envelope * np.sin(omega_n * t + phase)
        theta_clean += 0.1 * np.sin(3 * omega_n * t)  # Nonlinearity
        
        # Compute omega from clean signal (analytical derivative approximation)
        # d/dt[e^(-bt) * sin(wt)] = e^(-bt) * (w*cos(wt) - b*sin(wt))
        omega_clean = envelope * (omega_n * np.cos(omega_n * t + phase) 
                                  - damping * np.sin(omega_n * t + phase))
        omega_clean += 0.1 * 3 * omega_n * np.cos(3 * omega_n * t)  # Nonlinearity derivative
        
        # Add noise AFTER computing derivatives (independent noise for theta and omega)
        theta = theta_clean + noise_std * np.random.randn(n_samples)
        omega = omega_clean + noise_std * np.random.randn(n_samples)
        
        data[f'theta_{i+1}'] = theta
        data[f'omega_{i+1}'] = omega
    
    df = pd.DataFrame(data)
    df['target'] = df['theta_1']  # Primary target
    return df


def get_windows(df: pd.DataFrame, target_col: str, 
                context_len: int, horizon: int, n_samples: int,
                covariate_cols: Optional[List[str]] = None) -> List[Dict]:
    """
    Extract (context, target, covariates) windows for evaluation.
    Samples windows uniformly across the dataset.
    """
    total_len = context_len + horizon
    max_start = len(df) - total_len
    
    if max_start < n_samples:
        n_samples = max_start
    
    starts = np.linspace(0, max_start, n_samples, dtype=int)
    
    windows = []
    for start in starts:
        end_ctx = start + context_len
        end_tgt = end_ctx + horizon
        
        window = {
            'history': df[target_col].values[start:end_ctx],
            'actual': df[target_col].values[end_ctx:end_tgt],
            'timestamps': df['timestamp'].values[end_ctx:end_tgt],
        }
        
        if covariate_cols:
            window['covariates'] = df[covariate_cols].values[start:end_ctx]
            window['future_covariates'] = df[covariate_cols].values[end_ctx:end_tgt]
        
        windows.append(window)
    
    return windows

print("Data loaders defined.")

---
## Part 1: Visual Sample Test

Plot a single forecast to visually assess model behavior.

In [7]:
def visual_test(forecasters: List[BaseForecaster], 
                history: np.ndarray, actual: np.ndarray,
                title: str = "Forecast Comparison",
                history_plot_len: int = 100) -> None:
    """
    Visual comparison of forecasters on a single sample.
    
    Args:
        forecasters: List of forecaster instances
        history: Context window (1D array)
        actual: Ground truth for forecast period
        title: Plot title
        history_plot_len: How much history to show
    """
    horizon = len(actual)
    colors = plt.cm.tab10(np.linspace(0, 1, len(forecasters)))
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), 
                             gridspec_kw={'height_ratios': [3, 1]})
    
    # --- Top: Forecasts ---
    ax = axes[0]
    
    # Plot history
    hist_x = np.arange(-history_plot_len, 0)
    ax.plot(hist_x, history[-history_plot_len:], 'k-', lw=1.5, label='History', alpha=0.7)
    
    # Plot actual
    fut_x = np.arange(horizon)
    ax.plot(fut_x, actual, 'k--', lw=2, label='Actual', marker='o', markersize=3)
    
    # Plot each forecaster
    metrics_text = []
    for i, forecaster in enumerate(forecasters):
        try:
            point, quantiles = forecaster.predict(history, horizon)
            mae = mean_absolute_error(actual, point)
            
            ax.plot(fut_x, point, '-', color=colors[i], lw=2, 
                    label=f'{forecaster.name} (MAE={mae:.3f})')
            
            if quantiles and 0.1 in quantiles and 0.9 in quantiles:
                ax.fill_between(fut_x, quantiles[0.1], quantiles[0.9],
                                alpha=0.2, color=colors[i])
            
            metrics_text.append(f"{forecaster.name}: MAE={mae:.4f}")
        except Exception as e:
            print(f"Error with {forecaster.name}: {e}")
            metrics_text.append(f"{forecaster.name}: ERROR")
    
    ax.axvline(0, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel('Timestep (0 = forecast start)')
    ax.set_ylabel('Value')
    ax.set_title(title)
    ax.legend(loc='upper right')
    
    # --- Bottom: Error comparison ---
    ax2 = axes[1]
    
    for i, forecaster in enumerate(forecasters):
        try:
            point, _ = forecaster.predict(history, horizon)
            error = actual - point
            ax2.plot(fut_x, error, '-', color=colors[i], lw=1.5, 
                     label=forecaster.name, marker='.')
        except:
            pass
    
    ax2.axhline(0, color='black', ls='-', alpha=0.3)
    ax2.set_xlabel('Timestep')
    ax2.set_ylabel('Error (actual - pred)')
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()
    
    # Print metrics
    print("\n" + "="*50)
    print("METRICS SUMMARY")
    print("="*50)
    for m in metrics_text:
        print(f"  {m}")

print("Visual test function defined.")

Visual test function defined.


In [ ]:
# Run visual test on ETTh1
print("Loading ETTh1...")
df = load_etth1()
print(f"Loaded: {len(df)} rows, columns: {df.columns.tolist()}")

# Get one window
windows = get_windows(df, 'target', context_len=256, horizon=48, n_samples=1)
w = windows[0]

# Initialize forecasters
forecasters = [NaiveForecaster(), SeasonalNaiveForecaster(24)]

# TabPFN-TS
if MODELS_AVAILABLE.get('tabpfn_ts'):
    forecasters.append(TabPFNTSForecaster())

# Chronos-2
if MODELS_AVAILABLE.get('chronos'):
    try:
        forecasters.append(Chronos2Forecaster(model_name="amazon/chronos-bolt-small"))
    except Exception as e:
        print(f"Chronos-2 failed: {e}")

print(f"\nForecasters: {[f.name for f in forecasters]}")

# Run visual test
visual_test(forecasters, w['history'], w['actual'], 
            title="ETTh1: Oil Temperature Forecast (48h)")

---
## Part 2: Holistic Evaluation (GIFT-Eval Style)

Aggregate metrics across multiple samples, horizons, and settings.

**Methodology** (following [GIFT-Eval](https://arxiv.org/abs/2410.10393)):
1. Sample multiple windows from the dataset
2. Evaluate at multiple horizons
3. Aggregate with mean + std
4. Rank models by **rMAE** (relative to naive) and CRPS

In [ ]:
def holistic_eval(forecasters: List[BaseForecaster],
                  df: pd.DataFrame,
                  target_col: str = 'target',
                  covariate_cols: Optional[List[str]] = None,
                  context_len: int = 256,
                  horizons: List[int] = [24, 48, 96],
                  n_samples: int = 10,
                  seasonality: int = 24) -> pd.DataFrame:
    """
    GIFT-Eval style holistic evaluation.
    
    Returns DataFrame with metrics per (model, horizon).
    """
    results = []
    
    for horizon in horizons:
        print(f"\nHorizon = {horizon}...")
        windows = get_windows(df, target_col, context_len, horizon, n_samples, covariate_cols)
        
        for forecaster in forecasters:
            all_metrics = []
            
            for w in windows:
                try:
                    point, quantiles = forecaster.predict(
                        w['history'], horizon,
                        covariates=w.get('covariates'),
                        future_covariates=w.get('future_covariates')
                    )
                    
                    metrics = compute_all_metrics(
                        w['actual'], point, w['history'], 
                        quantiles, seasonality
                    )
                    all_metrics.append(metrics)
                except Exception as e:
                    print(f"  Error: {forecaster.name}: {e}")
            
            if all_metrics:
                # Aggregate
                agg = {}
                for key in all_metrics[0].keys():
                    vals = [m[key] for m in all_metrics if np.isfinite(m[key])]
                    if vals:
                        agg[f'{key}_mean'] = np.mean(vals)
                        agg[f'{key}_std'] = np.std(vals)
                
                results.append({
                    'model': forecaster.name,
                    'horizon': horizon,
                    'n_samples': len(all_metrics),
                    **agg
                })
    
    return pd.DataFrame(results)


def print_results_table(results_df: pd.DataFrame, 
                        metrics: List[str] = ['MAE', 'rMAE', 'CRPS']) -> None:
    """
    Pretty-print results table with rankings.
    
    Default metrics:
    - MAE: Absolute error (for scale reference)
    - rMAE: Relative to naive (< 1 = beats naive) <- THE KEY METRIC
    - CRPS: Probabilistic calibration
    """
    print("\n" + "="*80)
    print("HOLISTIC EVALUATION RESULTS")
    print("="*80)
    print("Key: rMAE < 1 means BETTER than naive, rMAE > 1 means WORSE than naive")
    
    for horizon in results_df['horizon'].unique():
        print(f"\n--- Horizon = {horizon} ---")
        subset = results_df[results_df['horizon'] == horizon].copy()
        
        # Compute ranks
        for m in metrics:
            col = f'{m}_mean'
            if col in subset.columns:
                subset[f'{m}_rank'] = subset[col].rank()
        
        # Print
        cols = ['model'] + [f'{m}_mean' for m in metrics if f'{m}_mean' in subset.columns]
        print(subset[cols].to_string(index=False, float_format='{:.4f}'.format))
    
    # Overall ranking by rMAE
    print("\n" + "="*80)
    print("OVERALL RANKING (by mean rMAE across horizons)")
    print("="*80)
    
    if 'rMAE_mean' in results_df.columns:
        ranking = results_df.groupby('model')['rMAE_mean'].mean().sort_values()
        for i, (model, rmae) in enumerate(ranking.items(), 1):
            status = "✓ beats naive" if rmae < 1 else "✗ worse than naive"
            print(f"  {i}. {model}: rMAE = {rmae:.4f} ({status})")

print("Holistic evaluation functions defined.")

In [ ]:
# ============================================================================
# VISUALIZATION FUNCTIONS
# ============================================================================

def plot_results_bar(results_df: pd.DataFrame, 
                     metric: str = 'MAE',
                     figsize: tuple = (12, 5)) -> None:
    """
    Bar chart comparing models across horizons.
    """
    col = f'{metric}_mean'
    std_col = f'{metric}_std'
    
    if col not in results_df.columns:
        print(f"Metric {metric} not found in results")
        return
    
    horizons = sorted(results_df['horizon'].unique())
    models = results_df['model'].unique()
    n_horizons = len(horizons)
    n_models = len(models)
    
    # Color palette
    colors = plt.cm.Set2(np.linspace(0, 1, n_models))
    
    fig, ax = plt.subplots(figsize=figsize)
    
    bar_width = 0.8 / n_models
    x = np.arange(n_horizons)
    
    for i, model in enumerate(models):
        model_data = results_df[results_df['model'] == model]
        vals = [model_data[model_data['horizon'] == h][col].values[0] 
                if len(model_data[model_data['horizon'] == h]) > 0 else 0 
                for h in horizons]
        
        # Error bars if available
        errs = None
        if std_col in results_df.columns:
            errs = [model_data[model_data['horizon'] == h][std_col].values[0] 
                    if len(model_data[model_data['horizon'] == h]) > 0 else 0 
                    for h in horizons]
        
        bars = ax.bar(x + i * bar_width, vals, bar_width, 
                      label=model, color=colors[i], yerr=errs, capsize=3)
        
        # Add value labels
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=8, rotation=0)
    
    ax.set_xlabel('Forecast Horizon', fontsize=12)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f'Model Comparison: {metric}', fontsize=14, fontweight='bold')
    ax.set_xticks(x + bar_width * (n_models - 1) / 2)
    ax.set_xticklabels([f'H={h}' for h in horizons])
    ax.legend(loc='upper left', framealpha=0.9)
    ax.grid(axis='y', alpha=0.3)
    
    # Add reference line for rMAE
    if metric == 'rMAE':
        ax.axhline(1.0, color='red', linestyle='--', alpha=0.7, label='Naive baseline')
        ax.text(0.02, 1.02, 'Beats naive ↓', transform=ax.get_yaxis_transform(), 
                fontsize=9, color='green')
    
    plt.tight_layout()
    plt.show()


def plot_results_heatmap(results_df: pd.DataFrame,
                         metrics: List[str] = ['MAE', 'rMAE', 'CRPS'],
                         figsize: tuple = (10, 6)) -> None:
    """
    Heatmap of metrics across models and horizons.
    """
    # Pivot for each metric
    fig, axes = plt.subplots(1, len(metrics), figsize=figsize)
    if len(metrics) == 1:
        axes = [axes]
    
    for ax, metric in zip(axes, metrics):
        col = f'{metric}_mean'
        if col not in results_df.columns:
            ax.set_title(f'{metric}: N/A')
            continue
        
        pivot = results_df.pivot(index='model', columns='horizon', values=col)
        
        # Normalize for color mapping (lower is better)
        im = ax.imshow(pivot.values, cmap='RdYlGn_r', aspect='auto')
        
        # Labels
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f'H={h}' for h in pivot.columns])
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_title(metric, fontsize=12, fontweight='bold')
        
        # Add values
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                color = 'white' if val > pivot.values.mean() else 'black'
                ax.text(j, i, f'{val:.3f}', ha='center', va='center', 
                        fontsize=9, color=color)
        
        plt.colorbar(im, ax=ax, shrink=0.8)
    
    plt.suptitle('Benchmark Results Heatmap', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


def plot_results_radar(results_df: pd.DataFrame,
                       horizon: int = None,
                       metrics: List[str] = ['MAE', 'rMAE', 'MASE', 'CRPS'],
                       figsize: tuple = (8, 8)) -> None:
    """
    Radar/spider chart comparing models across metrics.
    """
    if horizon is None:
        horizon = results_df['horizon'].iloc[0]
    
    subset = results_df[results_df['horizon'] == horizon]
    models = subset['model'].unique()
    
    # Filter to available metrics
    available_metrics = [m for m in metrics if f'{m}_mean' in subset.columns]
    n_metrics = len(available_metrics)
    
    if n_metrics < 3:
        print("Need at least 3 metrics for radar chart")
        return
    
    # Compute angles
    angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
    angles += angles[:1]  # Close the polygon
    
    fig, ax = plt.subplots(figsize=figsize, subplot_kw=dict(polar=True))
    colors = plt.cm.Set1(np.linspace(0, 1, len(models)))
    
    for i, model in enumerate(models):
        model_data = subset[subset['model'] == model]
        values = [model_data[f'{m}_mean'].values[0] for m in available_metrics]
        
        # Normalize values (0-1 scale, inverted so lower is better = further out)
        max_vals = [subset[f'{m}_mean'].max() for m in available_metrics]
        min_vals = [subset[f'{m}_mean'].min() for m in available_metrics]
        norm_values = [(max_v - v) / (max_v - min_v + 1e-8) 
                       for v, max_v, min_v in zip(values, max_vals, min_vals)]
        norm_values += norm_values[:1]  # Close polygon
        
        ax.plot(angles, norm_values, 'o-', linewidth=2, label=model, color=colors[i])
        ax.fill(angles, norm_values, alpha=0.15, color=colors[i])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(available_metrics, fontsize=10)
    ax.set_title(f'Model Comparison (H={horizon})\n(further out = better)', 
                 fontsize=12, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    
    plt.tight_layout()
    plt.show()


def plot_ranking_summary(results_df: pd.DataFrame, 
                         metric: str = 'rMAE',
                         figsize: tuple = (10, 4)) -> None:
    """
    Summary ranking plot with overall scores.
    """
    col = f'{metric}_mean'
    if col not in results_df.columns:
        print(f"Metric {metric} not found")
        return
    
    # Compute mean across horizons
    ranking = results_df.groupby('model')[col].mean().sort_values()
    
    fig, ax = plt.subplots(figsize=figsize)
    
    colors = ['#2ecc71' if v < 1 else '#e74c3c' for v in ranking.values] if metric == 'rMAE' else plt.cm.viridis(np.linspace(0.3, 0.9, len(ranking)))
    
    bars = ax.barh(range(len(ranking)), ranking.values, color=colors)
    ax.set_yticks(range(len(ranking)))
    ax.set_yticklabels(ranking.index)
    ax.set_xlabel(f'Mean {metric} (across horizons)', fontsize=11)
    ax.set_title(f'Overall Ranking by {metric}', fontsize=14, fontweight='bold')
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, ranking.values)):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10)
    
    # Reference line for rMAE
    if metric == 'rMAE':
        ax.axvline(1.0, color='red', linestyle='--', alpha=0.7)
        ax.text(1.02, len(ranking) - 0.5, '← Better than naive', fontsize=9, color='green')
    
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_all_results(results_df: pd.DataFrame, metrics: List[str] = ['MAE', 'rMAE', 'CRPS']):
    """Master function to plot all result visualizations."""
    print("\n" + "="*70)
    print("BENCHMARK RESULTS VISUALIZATION")
    print("="*70)
    
    # 1. Bar charts
    for metric in metrics:
        if f'{metric}_mean' in results_df.columns:
            plot_results_bar(results_df, metric)
    
    # 2. Heatmap
    available = [m for m in metrics if f'{m}_mean' in results_df.columns]
    if available:
        plot_results_heatmap(results_df, available)
    
    # 3. Radar (for first horizon)
    if len(available) >= 3:
        plot_results_radar(results_df, metrics=available)
    
    # 4. Overall ranking
    plot_ranking_summary(results_df, 'rMAE' if 'rMAE_mean' in results_df.columns else 'MAE')

print("Visualization functions defined:")
print("  - plot_results_bar(): Bar chart by horizon")
print("  - plot_results_heatmap(): Heatmap of all metrics")
print("  - plot_results_radar(): Radar/spider chart")
print("  - plot_ranking_summary(): Overall ranking")
print("  - plot_all_results(): All visualizations")

In [ ]:
# Run holistic evaluation
print("Running holistic evaluation...")
print(f"Config: horizons={CFG.horizons}, n_samples={CFG.n_samples}")

# Build forecaster list dynamically
forecasters = [NaiveForecaster(), SeasonalNaiveForecaster(24)]

if MODELS_AVAILABLE.get('tabpfn_ts'):
    forecasters.append(TabPFNTSForecaster())

if MODELS_AVAILABLE.get('chronos'):
    try:
        forecasters.append(Chronos2Forecaster(model_name="amazon/chronos-bolt-small"))
    except:
        pass

print(f"Testing: {[f.name for f in forecasters]}")

results = holistic_eval(
    forecasters=forecasters,
    df=df,
    target_col='target',
    covariate_cols=['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL'],
    context_len=CFG.context_length,
    horizons=CFG.horizons,
    n_samples=CFG.n_samples,
    seasonality=24
)

print_results_table(results)

In [ ]:
# Plot all results nicely
plot_all_results(results, metrics=['MAE', 'rMAE', 'CRPS'])

---
## Part 3: Ablations

Quick ablation experiments to test specific hypotheses.

In [ ]:
# Ablation: With vs Without Covariates
print("Ablation: Effect of Covariates")
print("="*50)

if MODELS_AVAILABLE.get('tabpfn_ts'):
    forecaster = TabPFNTSForecaster()
    covariate_cols = ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL']
    
    # With covariates
    results_cov = holistic_eval(
        [forecaster], df, 'target', covariate_cols,
        context_len=256, horizons=[48], n_samples=5
    )
    
    # Without covariates
    results_nocov = holistic_eval(
        [forecaster], df, 'target', None,
        context_len=256, horizons=[48], n_samples=5
    )
    
    print(f"\nWith covariates:    MAE = {results_cov['MAE_mean'].values[0]:.4f}")
    print(f"Without covariates: MAE = {results_nocov['MAE_mean'].values[0]:.4f}")
    delta = (results_nocov['MAE_mean'].values[0] - results_cov['MAE_mean'].values[0]) / results_nocov['MAE_mean'].values[0] * 100
    print(f"Improvement from covariates: {delta:+.1f}%")
    
    # Nice comparison plot
    fig, ax = plt.subplots(figsize=(8, 4))
    x = ['With Covariates', 'Without Covariates']
    vals = [results_cov['MAE_mean'].values[0], results_nocov['MAE_mean'].values[0]]
    colors = ['#2ecc71', '#e74c3c']
    bars = ax.bar(x, vals, color=colors)
    ax.set_ylabel('MAE')
    ax.set_title('TabPFN-TS: Effect of Covariates (H=48)', fontweight='bold')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.3f}', ha='center')
    plt.tight_layout()
    plt.show()
else:
    print("TabPFN-TS not available for ablation.")

In [ ]:
# Ablation: Context Length
print("\nAblation: Effect of Context Length")
print("="*50)

context_lengths = [64, 128, 256, 512]
context_results = []

for ctx_len in context_lengths:
    res = holistic_eval(
        [NaiveForecaster(), SeasonalNaiveForecaster(24)],
        df, 'target', None,
        context_len=ctx_len, horizons=[48], n_samples=5
    )
    res['context_len'] = ctx_len
    context_results.append(res)

context_df = pd.concat(context_results)
pivot_table = context_df[['model', 'context_len', 'MAE_mean']].pivot(
    index='context_len', columns='model', values='MAE_mean'
)
print(pivot_table)

# Plot context length effect
fig, ax = plt.subplots(figsize=(10, 5))
for model in pivot_table.columns:
    ax.plot(pivot_table.index, pivot_table[model], 'o-', label=model, linewidth=2, markersize=8)

ax.set_xlabel('Context Length', fontsize=12)
ax.set_ylabel('MAE', fontsize=12)
ax.set_title('Effect of Context Length on Forecast Accuracy', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xticks(context_lengths)
plt.tight_layout()
plt.show()

---
## Part 4: Synthetic Pendulum Test

Test on synthetic N-link pendulum data (preview of Unify sanity check).

In [ ]:
# Generate 3-link pendulum data
pend_df = load_synthetic_pendulum(n_links=3, n_samples=5000)
print(f"Generated: {pend_df.shape}")
print(f"Columns: {pend_df.columns.tolist()}")

# Plot sample
fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
plot_range = 500

for i, ax in enumerate(axes):
    ax.plot(pend_df[f'theta_{i+1}'][:plot_range], label=f'θ_{i+1}')
    ax.plot(pend_df[f'omega_{i+1}'][:plot_range], label=f'ω_{i+1}', alpha=0.7)
    ax.legend(loc='upper right')
    ax.set_ylabel(f'Link {i+1}')

axes[0].set_title('Synthetic 3-Link Pendulum')
axes[-1].set_xlabel('Timestep')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on pendulum
windows = get_windows(pend_df, 'target', context_len=256, horizon=50, n_samples=1)
w = windows[0]

visual_test(
    [NaiveForecaster(), SeasonalNaiveForecaster(50)],
    w['history'], w['actual'],
    title="Synthetic 3-Link Pendulum: θ₁ Forecast"
)

---
## Summary & Next Steps

### This Notebook Provides:
1. **Visual test:** Quick sanity check on single sample
2. **Holistic metrics:** GIFT-eval style aggregated evaluation
3. **Model wrappers:** Unified interface for TabPFN, Chronos, baselines
4. **Ablations:** Context length, covariates

### To Extend:
```python
# Add new model
class MyForecaster(BaseForecaster):
    name = "my_model"
    def predict(self, history, horizon, **kwargs):
        # Your implementation
        return point, quantiles

# Add new dataset
def load_my_dataset() -> pd.DataFrame:
    # Return df with 'timestamp', 'target', optional covariates
    pass
```

### Key Findings:
- [ ] Fill in after running experiments

### Links:
- [GIFT-Eval Paper](https://arxiv.org/abs/2410.10393)
- [TabPFN-TS](https://github.com/PriorLabs/tabpfn-time-series)
- [Chronos-2](https://github.com/amazon-science/chronos-forecasting)
- [GluonTS Metrics](https://ts.gluon.ai/stable/api/gluonts/gluonts.ev.metrics.html)